**AlphaGenome profiling: tasks 3 to 9**

Weights and genome in `/content/ag_data`, outputs in `experiments/results/`. `AG_FP32=1` in cell 1 on gpus without bf16 (T4).

**1. Setup**

Pulls the branch, installs the port, prints the commit and gpu.

In [ ]:
%env AG_DATA=/content/ag_data
%env HF_HOME=/content/ag_data/hf
%env AG_FP32=0
!git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag 2>/dev/null || git -C /content/ag pull -q
%cd /content/ag/experiments
!pip install -q -e .. pyfaidx huggingface_hub
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

**2. Weights**

Downloads `model_all_folds.safetensors`.

In [ ]:
!python -c "import common; p = common.fetch_weights(); print(p, p.stat().st_size >> 20, 'MB')"

**3. Reference genome**

Downloads GRCh38.p13 and its index.

In [ ]:
!mkdir -p $AG_DATA/ref
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!test -s $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

**4. Task 4: fixed input**

`chr19:40480552-41529128`, 1 Mb centred on CYP2B6. Writes `inputs/cyp2b6_1m.json` and `inputs/cyp2b6_1m_onehot.npy`.

In [ ]:
!python input.py
!cat inputs/cyp2b6_1m.json

**5. Task 3: forward pass at every length**

2 kb, 16 kb, 131 kb, 1 Mb: output shapes, nan/inf, value ranges, centre agreement.

In [ ]:
!python lengths.py

**6. Task 5: end-to-end sweep**

Median latency and peak memory per length, eager and `torch.compile`. Writes `results/sweep_<gpu>.csv`.

In [ ]:
!python sweep.py

**7. Task 6: profiling harness**

One profiled eager forward per length. Writes `results/trace_<length>.json.gz` and `results/modules_<length>.tsv`.

In [ ]:
!python trace.py

**8. Task 7: kernel-to-module attribution**

Kernel time per module and per stage. Writes `results/kernels_<length>.csv` and `results/stages_<length>.csv`.

In [ ]:
!python attribute.py

**9. Task 8: stage share and operator mix**

Writes `results/stage_share.png` and `results/op_mix.png`.

In [ ]:
!python plots.py
from IPython.display import Image, display
display(Image("results/stage_share.png"))
display(Image("results/op_mix.png"))

**10. Task 9: parallelism per stage**

Writes `results/parallel_<length>.csv`.

In [ ]:
!python parallel.py

**11. Download**

Zips `results/` and the input manifest.

In [ ]:
!zip -qr results.zip results inputs/cyp2b6_1m.json
from google.colab import files
files.download("results.zip")